# Multi-Source News Credibility Checker (Simple)

This notebook implements a **simple agent workflow** in LangGraph:
1. Router Agent
2. Parallel Agent (multi-source web search + perspective extraction)
3. Critic Agent (finds contradictions)
4. Aggregator Agent (builds unified analysis)
5. Scoring Agent (credibility confidence)
6. Final Output

It uses:
- **OpenAI SDK** (`openai`) for the router step
- **LangChain + LangGraph** for orchestration and analysis


In [22]:
# If needed, run this once:
!pip install -q openai langchain langchain-openai langgraph duckduckgo-search pydantic openai-agents

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  return process_handler(cmd, _system_body)
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  return process_handler(cmd, _system_body)
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  return process_handler(cmd, _system_body)


In [23]:
import os
import json
from typing import TypedDict, List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from pydantic import BaseModel, Field
from duckduckgo_search import DDGS

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# Prefer OpenAI Agents SDK function_tool; fallback keeps notebook runnable.
try:
    from agents import function_tool
except Exception:
    def function_tool(func=None, **kwargs):
        if func is None:
            def wrapper(f):
                return f
            return wrapper
        return func

In [ ]:
# Set your key in environment before running this notebook:
os.environ["OPENAI_API_KEY"] = 

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) #gpt-4.1-mini

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Hello"}
    ],
    max_tokens=10
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [25]:
# ---------- Structured output schemas ----------
class RouterOutput(BaseModel):
    topic: str = Field(description="Main topic of claim")
    search_queries: List[str] = Field(description="3 focused search queries")

class PerspectiveOutput(BaseModel):
    source_name: str
    url: str
    headline: str
    stance: str = Field(description="support / contradict / mixed / unclear")
    key_points: List[str]

class CriticOutput(BaseModel):
    contradictions: List[str]
    notes: List[str]

class ScoreOutput(BaseModel):
    confidence_score: int = Field(description="0 to 100")
    rationale: str

In [26]:
# ---------- State ----------
class NewsState(TypedDict, total=False):
    claim: str
    topic: str
    queries: List[str]
    raw_results: List[Dict[str, Any]]
    perspectives: List[Dict[str, Any]]
    contradictions: List[str]
    critic_notes: List[str]
    aggregated_analysis: str
    confidence_score: int
    score_rationale: str
    final_report: str

In [27]:
# ---------- Helpers ----------
def _web_search_impl(query: str, max_results: int = 5) -> List[Dict[str, Any]]:
    rows = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            rows.append({
                "query": query,
                "title": r.get("title", ""),
                "href": r.get("href", ""),
                "body": r.get("body", ""),
                "source": (r.get("href", "").split("/")[2] if r.get("href") else "unknown")
            })
    return rows


@function_tool
def web_search_tool(query: str, max_results: int = 5) -> str:
    """Search the web and return JSON list of results (title, href, snippet, source)."""
    rows = _web_search_impl(query=query, max_results=max_results)
    return json.dumps(rows, ensure_ascii=False)


def run_web_search_tool(query: str, max_results: int = 5) -> List[Dict[str, Any]]:
    """Tool runner that works with both real @function_tool and fallback decorator."""
    try:
        out = web_search_tool(query=query, max_results=max_results)
        if isinstance(out, str):
            return json.loads(out)
    except Exception:
        pass
    return _web_search_impl(query=query, max_results=max_results)


perspective_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a neutral fact-checking analyst. Return concise structured output only."),
    ("human",      "Claim: {claim}\n"
     "Source URL: {url}\n"
     "Title: {title}\n"
     "Snippet: {snippet}\n"
     "Classify stance and extract key points.")
])

perspective_chain = perspective_prompt | llm.with_structured_output(PerspectiveOutput)


critic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict critic. Detect contradictions and weak evidence across sources."),
    ("human",      "Claim: {claim}\n"
     "Perspectives JSON:\n{perspectives_json}\n"
     "Find contradictions and quality concerns.")
])

critic_chain = critic_prompt | llm.with_structured_output(CriticOutput)


aggregate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You aggregate source perspectives into one balanced summary."),
    ("human",      "Claim: {claim}\n"
     "Perspectives:\n{perspectives_json}\n"
     "Contradictions:\n{contradictions_json}\n"
     "Write a short balanced synthesis with evidence mentions.")
])


score_prompt = ChatPromptTemplate.from_messages([
    ("system", "You assign a credibility confidence score from 0-100."),
    ("human",      "Claim: {claim}\n"
     "Aggregated analysis:\n{aggregated_analysis}\n"
     "Contradictions:\n{contradictions_json}\n"
     "Give score and rationale.")
])

score_chain = score_prompt | llm.with_structured_output(ScoreOutput)

In [28]:
# ---------- Agents (nodes) ----------
def router_agent(state: NewsState) -> NewsState:
    """Uses OpenAI SDK directly to generate topic + search queries."""
    claim = state["claim"]

    resp = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "system",
                "content": "You create web search plans for fact-checking. Return JSON only."
            },
            {
                "role": "user",
                "content": (
                    "Claim: " + claim + "\n"
                    "Return JSON with keys: topic, search_queries (exactly 3 strings)."
                )
            }
        ]
    )

    text = resp.output_text.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        text = text.replace("json", "", 1).strip()
    parsed = json.loads(text)

    # Validate lightly with pydantic
    out = RouterOutput(**parsed)
    return {"topic": out.topic, "queries": out.search_queries}


def parallel_agent(state: NewsState) -> NewsState:
    """Runs search in parallel and extracts per-source perspective."""
    claim = state["claim"]
    queries = state["queries"]

    # 1) Search all queries in parallel
    all_rows: List[Dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=3) as ex:
        future_map = {ex.submit(run_web_search_tool, q, 4): q for q in queries}
        for fut in as_completed(future_map):
            all_rows.extend(fut.result())

    # De-duplicate by URL and keep top 10
    seen = set()
    dedup = []
    for r in all_rows:
        u = r.get("href", "")
        if u and u not in seen:
            seen.add(u)
            dedup.append(r)
        if len(dedup) >= 10:
            break

    # 2) Perspective extraction in parallel
    perspectives: List[Dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=5) as ex:
        future_map = {
            ex.submit(
                perspective_chain.invoke,
                {
                    "claim": claim,
                    "url": r.get("href", ""),
                    "title": r.get("title", ""),
                    "snippet": r.get("body", "")
                }
            ): r
            for r in dedup
        }
        for fut in as_completed(future_map):
            try:
                p = fut.result()
                perspectives.append(p.model_dump())
            except Exception:
                # Keep flow simple and robust
                r = future_map[fut]
                perspectives.append({
                    "source_name": r.get("source", "unknown"),
                    "url": r.get("href", ""),
                    "headline": r.get("title", ""),
                    "stance": "unclear",
                    "key_points": [r.get("body", "")[:180]]
                })

    return {"raw_results": dedup, "perspectives": perspectives}


def critic_agent(state: NewsState) -> NewsState:
    out = critic_chain.invoke({
        "claim": state["claim"],
        "perspectives_json": json.dumps(state["perspectives"], ensure_ascii=False)
    })
    return {
        "contradictions": out.contradictions,
        "critic_notes": out.notes
    }


def aggregator_agent(state: NewsState) -> NewsState:
    msg = aggregate_prompt.invoke({
        "claim": state["claim"],
        "perspectives_json": json.dumps(state["perspectives"], ensure_ascii=False),
        "contradictions_json": json.dumps(state.get("contradictions", []), ensure_ascii=False)
    })
    summary = llm.invoke(msg).content
    return {"aggregated_analysis": summary}


def scoring_agent(state: NewsState) -> NewsState:
    out = score_chain.invoke({
        "claim": state["claim"],
        "aggregated_analysis": state["aggregated_analysis"],
        "contradictions_json": json.dumps(state.get("contradictions", []), ensure_ascii=False)
    })
    return {
        "confidence_score": out.confidence_score,
        "score_rationale": out.rationale
    }


def final_output_agent(state: NewsState) -> NewsState:
    report = {
        "claim": state["claim"],
        "topic": state.get("topic", ""),
        "queries": state.get("queries", []),
        "num_sources": len(state.get("perspectives", [])),
        "contradictions": state.get("contradictions", []),
        "analysis": state.get("aggregated_analysis", ""),
        "confidence_score": state.get("confidence_score", 0),
        "score_rationale": state.get("score_rationale", "")
    }
    return {"final_report": json.dumps(report, indent=2, ensure_ascii=False)}

In [29]:
# ---------- Build LangGraph ----------
graph_builder = StateGraph(NewsState)

graph_builder.add_node("router_agent", router_agent)
graph_builder.add_node("parallel_agent", parallel_agent)
graph_builder.add_node("critic_agent", critic_agent)
graph_builder.add_node("aggregator_agent", aggregator_agent)
graph_builder.add_node("scoring_agent", scoring_agent)
graph_builder.add_node("final_output_agent", final_output_agent)

graph_builder.set_entry_point("router_agent")
graph_builder.add_edge("router_agent", "parallel_agent")
graph_builder.add_edge("parallel_agent", "critic_agent")
graph_builder.add_edge("critic_agent", "aggregator_agent")
graph_builder.add_edge("aggregator_agent", "scoring_agent")
graph_builder.add_edge("scoring_agent", "final_output_agent")
graph_builder.add_edge("final_output_agent", END)

graph = graph_builder.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


## LangGraph Flowchart

```mermaid
flowchart TD
    START([START]) --> router_agent
    router_agent --> parallel_agent
    parallel_agent --> critic_agent
    critic_agent --> aggregator_agent
    aggregator_agent --> scoring_agent
    scoring_agent --> final_output_agent
    final_output_agent --> RESULT([END / RESULT])
```


In [30]:
# Optional: auto-generate graph Mermaid from compiled LangGraph
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router_agent(router_agent)
	parallel_agent(parallel_agent)
	critic_agent(critic_agent)
	aggregator_agent(aggregator_agent)
	scoring_agent(scoring_agent)
	final_output_agent(final_output_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> router_agent;
	aggregator_agent --> scoring_agent;
	critic_agent --> aggregator_agent;
	parallel_agent --> critic_agent;
	router_agent --> parallel_agent;
	scoring_agent --> final_output_agent;
	final_output_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [34]:
graph.invoke({"claim": "The WHO has declared a new global public health emergency in 2026"})

C:\Users\Administrator\AppData\Local\Temp\1\ipykernel_6504\375818690.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\Administrator\AppData\Local\Temp\1\ipykernel_6504\375818690.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\Administrator\AppData\Local\Temp\1\ipykernel_6504\375818690.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PerspectiveOutput(source_...ovided in the source.']), input_type=PerspectiveOutput])
  return se

{'claim': 'The WHO has declared a new global public health emergency in 2026',
 'topic': 'WHO global public health emergency 2026',
 'queries': ['WHO global public health emergency announcement 2026',
  '2026 WHO health emergency declaration news',
  'latest WHO public health emergencies 2026'],
 'raw_results': [{'query': 'WHO global public health emergency announcement 2026',
   'title': 'In charts: 7 global shifts defining 2025 so far | World Economic Forum',
   'href': 'https://www.weforum.org/stories/2025/08/inflection-points-7-global-shifts-defining-2025-so-far-in-charts/',
   'body': 'Aug 5, 2025 · So far, this year has been marked by significant global shifts, including increased geopolitical instability, the accelerating impact of AI and a changing labour market. Economic factors …',
   'source': 'www.weforum.org'},
  {'query': 'WHO global public health emergency announcement 2026',
   'title': 'Global Risks Report 2026 - The World Economic Forum',
   'href': 'https://www.wefor

In [33]:
# ---------- Run ----------
#claim = "A new law was passed this week that completely bans AI-generated content in U.S. schools."
#claim = "Based on the ongoing tension between Israel and Iran, will it possible for World War 3 to happen soon ?"
#claim = "The WHO has declared a new global public health emergency in 2026"
claim = "I have two claims - The WHO has declared a new global public health emergency in 2026 and A new law was passed this week that completely bans AI-generated content in U.S. schools."


result = graph.invoke({"claim": claim})
print(result["final_report"])

C:\Users\Administrator\AppData\Local\Temp\1\ipykernel_6504\375818690.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\Administrator\AppData\Local\Temp\1\ipykernel_6504\375818690.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\Administrator\AppData\Local\Temp\1\ipykernel_6504\375818690.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PerspectiveOutput(source_...tent in U.S. schools.']), input_type=PerspectiveOutput])
  return se

{
  "claim": "I have two claims - The WHO has declared a new global public health emergency in 2026 and A new law was passed this week that completely bans AI-generated content in U.S. schools.",
  "topic": "Fact-checking global public health emergency and AI content ban in schools",
  "queries": [
    "WHO declares global public health emergency 2026",
    "new law AI-generated content ban U.S. schools October 2023",
    "public health emergencies declared by WHO history"
  ],
  "num_sources": 4,
  "contradictions": [],
  "analysis": "The claims regarding a new global public health emergency declared by the WHO in 2026 and a recent law banning AI-generated content in U.S. schools are not supported by the sources reviewed. All sources, including Google Help and Stack Overflow, do not provide any information related to these claims. Specifically, Google Help articles focus on Gmail account creation and sign-in processes, while Stack Overflow discussions center around JavaScript programm

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ScoreOutput(confidence_sc...alidity of the claims.'), input_type=ScoreOutput])
  return self.__pydantic_serializer__.to_python(


## Notes
- This is intentionally simple (good for learning/demo).
- For production use, add:
  - Source reliability weighting (domain trust list)
  - Better contradiction detection with citation spans
  - Caching and retry logic
  - Evaluation set for score calibration
